# Anomaly detection with Recurrent Neural Networks
In this notebook, we will implement a basic RNN for anomaly detection. The RNN consists of a single recurrent layer and is trained to predict the next value of a time series. 

If the RNN is trained with normal data, it will be able to predict the next normal value of a time series. When used to detect anomalies, this prediction can be compared with the actual `next value` to determine whether it is normal or not.

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Flatten, Input, LSTM
from keras.losses import mean_squared_error
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
HIDDED_UNITS = 3


# Univariate time series

In [ ]:
def generate_time_series(batch_size, n_steps, n_series_per_sample=1, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    total_series = batch_size * n_series_per_sample
    freq1, freq2, offsets1, offsets2 = rng.random((4, total_series, 1))
    time = np.linspace(0, 1, n_steps)

    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20))
    series += 0.1 * (rng.random((total_series, n_steps)) - 0.5)

    series = series.reshape(batch_size, n_steps, n_series_per_sample)
    return series.astype(np.float32)

# Sequence-to-vector network
Each time series consists of *n_steps* univariate time steps (e.g., the value of a single sensor), while its label is the *n_steps+1* time step. The idea is to train an RNN that predicts the next value of a time series, given the previous *n_steps* values.

In [ ]:
n_steps = 50 
n_series_per_sample = 1
rng = np.random.default_rng(SEED)
series = generate_time_series(10000, n_steps + 1,n_series_per_sample, rng=rng) 
X_train, y_train = np.squeeze(series[:7000, :n_steps], axis=2), series[:7000, -1] 
X_valid, y_valid = np.squeeze(series[7000:9000, :n_steps], axis=2), series[7000:9000, -1] 
X_test, y_test = np.squeeze(series[9000:, :n_steps], axis=2), series[9000:, -1]

# Generate time axis
time = np.linspace(0, n_steps+1, n_steps+1)

# Plot all time series in the first sample
plt.figure(figsize=(12, 6))
for j in range(n_series_per_sample):
    plt.plot(time, series[0, :, j], label=f'Series {j} in Sample 0')
plt.title(f'First Sample with {n_series_per_sample} Time Series')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Simple RNN
The next cell implements and trains a Simple RNN, with a single-neuron Recurrent layer and a final Dense layer with *linear* activation function for predicting the next value.

| <img src="./univariate-rnn.jpg" width="100%">  |
|--|
| A simple RNN with ```m``` hidden neurons. |

$$\mathbf{h}^{(0)} = 0$$
$$\mathbf{h}^{(t)} = \phi(W_{x}^{\top}\mathbf{x}^{(t)} + W_{h}^{\top}\mathbf{h}^{(t-1)} + \mathbf{b}^{(1)})\quad t\in[1,s]$$
$$\hat{y} = \sigma(W_{y}^{\top} \mathbf{h}^{(s)} + \mathbf{b}^{(2)})$$
where $\phi$ is the activation function of the recurrent layer (ReLU in this case), $\sigma$ is the activation function of the output layer (linear in this case), $s$ is the number of time steps, $x^{(t)}$ is the input at time step $t$, $h^{(t)}$ is the hidden state at time step $t$, and $\hat{y}$ is the predicted output.

In [ ]:
model = Sequential([ 
    Input(shape=(n_steps, 1)), #n_steps time steps, 1 feature
    SimpleRNN(HIDDED_UNITS,activation='relu'),
    Dense(1, activation='linear') 
])
print (model.summary())

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
early_stopping = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, restore_best_weights=True)
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_valid,y_valid),callbacks=[early_stopping])

# Prediction on the test set and comparison between the predicted and actual values

In [ ]:
y_pred = model.predict(X_test)
print ("Test MSE:", np.mean(mean_squared_error(y_test, y_pred)))

actual = np.asarray(y_test)
predicted = np.asarray(y_pred)

# Treat single-series arrays with shape (n_samples,) like (n_samples, 1)
if actual.ndim == 1:
    actual = actual[:, None]
if predicted.ndim == 1:
    predicted = predicted[:, None]

assert actual.shape == predicted.shape, (
    f"Shape mismatch: y_test {actual.shape}, y_pred {predicted.shape}"
)

n = min(200, len(actual))  # show the first 200 test examples
fig, axes = plt.subplots(actual.shape[1], 1, figsize=(20, 3 * actual.shape[1]), squeeze=False)

for i, ax in enumerate(axes[:, 0]):
    ax.plot(actual[:n, i], color="blue", label="Actual")
    ax.plot(predicted[:n, i], color="orange", label="Predicted", alpha=0.8)
    ax.set_title(f"Series {i + 1}")
    ax.set_xlabel("Test example")
    ax.set_ylabel("Value")
    ax.legend()

plt.tight_layout()
plt.show()

# Long-Short Memory Term (LSTM)
The next cell implements and trains an LSTM model, with a Recurrent layer with ```HIDDEN_UNITS``` units and a final Dense layer with *linear* activation function for predicting the next value.

In [ ]:

lstm_model = Sequential([ 
    Input(shape=(n_steps, 1)), #n_steps time steps
    LSTM(HIDDED_UNITS),
    Dense(1, activation='linear') 
])
print (lstm_model.summary())

# Compile the model
lstm_model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
early_stopping = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, restore_best_weights=True)
lstm_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_valid,y_valid),callbacks=[early_stopping])

# Prediction on the test set and comparison between the predicted and actual values

In [ ]:
# Prediction
y_pred = lstm_model.predict(X_test)
print (np.mean(mean_squared_error(y_test, y_pred)))

actual = np.asarray(y_test)
predicted = np.asarray(y_pred)

# Treat single-series arrays with shape (n_samples,) like (n_samples, 1)
if actual.ndim == 1:
    actual = actual[:, None]
if predicted.ndim == 1:
    predicted = predicted[:, None]

assert actual.shape == predicted.shape, (
    f"Shape mismatch: y_test {actual.shape}, y_pred {predicted.shape}"
)

n = min(200, len(actual))  # show the first 200 test examples
fig, axes = plt.subplots(actual.shape[1], 1, figsize=(20, 3 * actual.shape[1]), squeeze=False)

for i, ax in enumerate(axes[:, 0]):
    ax.plot(actual[:n, i], color="blue", label="Actual")
    ax.plot(predicted[:n, i], color="orange", label="Predicted", alpha=0.8)
    ax.set_title(f"Series {i + 1}")
    ax.set_xlabel("Test example")
    ax.set_ylabel("Value")
    ax.legend()

plt.tight_layout()
plt.show()